# Week 2: Acquisition-Rule Comparison

Week 2 keeps the thresholded-Branin active-learning loop fixed and changes only the acquisition rule. This notebook is intentionally small: the main implementation lives in `src/week2_acquisition_comparison.py`, and this notebook loads or runs that script, then displays the main Week 2 outputs.

In [ ]:
from pathlib import Path
import csv
import json
import sys

from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

OUTPUT_DIR = ROOT / "outputs" / "week2_acquisition_comparison"
OUTPUT_DIR

## Run or Load the Experiment

If `summary.json` is already present, the notebook loads the saved results. If not, it runs the Week 2 script once. The default run uses seeds 0 through 4 so the comparison is averaged over five initial designs.

In [ ]:
summary_path = OUTPUT_DIR / "summary.json"
if not summary_path.exists():
    from src.week2_acquisition_comparison import run_experiment
    run_experiment(output_dir=OUTPUT_DIR)

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("Experiment:", summary["experiment"])
print("Threshold:", f"{summary['threshold']:.6f}")
print("Seeds:", summary["seeds"])
print("Best method:", summary["best_final_method_by_mean_final_error"])
print("Fairness checks:", summary["fairness_checks_same_initial_indices_per_seed"])

## Main Result: Error Curves

The y-axis is the fraction of 4,000 exact Branin test labels predicted incorrectly. Lower is better. The shaded bands show seed-to-seed variation.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "error_curves_all_methods.png")))

## Final Error at Budget 50

This bar chart summarizes the final test error for each acquisition rule after 50 labelled Branin evaluations.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "final_error_bar_chart.png")))

## Where Each Rule Queried

For seed 0, this plot shows whether a method spent its budget near the exact Branin boundary or away from it. The stars are the shared initial six labels.

In [ ]:
display(Image(filename=str(OUTPUT_DIR / "query_locations_seed0.png")))

## Summary Tables

`method_summary_table.csv` gives one row per acquisition rule. `selected_budget_table.csv` gives errors at budgets 6, 20, and 50.

In [ ]:
def csv_as_markdown(path, max_rows=None):
    rows = list(csv.DictReader(path.open(newline="", encoding="utf-8")))
    if max_rows is not None:
        rows = rows[:max_rows]
    headers = list(rows[0].keys()) if rows else []
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        lines.append("| " + " | ".join(row[h] for h in headers) + " |")
    return "\n".join(lines)

display(Markdown("### Method Summary\n" + csv_as_markdown(OUTPUT_DIR / "method_summary_table.csv")))
display(Markdown("### Selected Budgets\n" + csv_as_markdown(OUTPUT_DIR / "selected_budget_table.csv")))

## Snapshot Figures

The snapshot figures show the GP's believed boundary `mu = 0` at 6, 20, and 50 labels. The dashed reference is the exact thresholded-Branin boundary.

In [ ]:
best_method = summary["best_final_method_by_mean_final_error"]
display(Image(filename=str(OUTPUT_DIR / f"snapshots_best_{best_method}_seed0.png")))
display(Image(filename=str(OUTPUT_DIR / "snapshots_straddle_seed0.png")))